# 📈 Fase 4 — Modelado · PANEL 3 (Pronóstico)
### Trabajo Final · Minería de Datos · UNMSM-FISI · 2026-I

**Pregunta:** ¿cuántos accidentes de trabajo se notificarán en los próximos meses?

---
### Checklist del requisito (imagen del profe)
| Contenido mínimo | ✔ |
|---|---|
| Serie temporal graficada **con tendencia** | ✅ (descomposición) |
| Pronóstico **≥ 4 períodos** siguientes | ✅ (6 meses) |
| **MAPE y RMSE reportados** | ✅ **obligatorio: visibles en el panel** |
| Modelo: media móvil / suavizado exp. / ARIMA / Prophet | ✅ **los 4 comparados** |
| Librerías: statsmodels · pandas rolling · Plotly | ✅ |

> **Nota:** el *concept drift* del Panel 2 **no afecta aquí**. Ahí clasificábamos la severidad
> (y el MTPE cambió ese criterio en 2023); aquí solo **contamos accidentes por mes**, y ese conteo
> es consistente en toda la serie. → **Usamos los 142 meses completos (2012–2024).**


In [14]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

GRANATE, DORADO, AZUL, VERDE, GRIS = "#7a1128", "#d4a72c", "#3b6ea5", "#2e7d5b", "#6b6b6b"
TEMPLATE = "plotly_white"

df = pd.read_csv("../data set/limpio.csv")
df["PERIODO"] = pd.to_datetime(dict(year=df["ANIOS"].astype(int),
                                    month=df["MES_N"].astype(int), day=1))
serie = df.groupby("PERIODO").size().sort_index()
serie = serie.asfreq("MS")                      # frecuencia mensual explícita
serie = serie.interpolate()                     # por si faltara algún mes

print(f"Serie mensual: {len(serie)} meses | {serie.index.min():%Y-%m} → {serie.index.max():%Y-%m}")
print(f"Media: {serie.mean():.0f} accidentes/mes | mín: {serie.min():.0f} | máx: {serie.max():.0f}")


Serie mensual: 149 meses | 2012-01 → 2024-05
Media: 2006 accidentes/mes | mín: 477 | máx: 3525


---
## 1️⃣ La serie temporal *(requisito: graficada con tendencia)*

In [15]:
tendencia = serie.rolling(12, center=True).mean()      # media móvil 12m = tendencia

fig = go.Figure()
fig.add_trace(go.Scatter(x=serie.index, y=serie.values, name="Accidentes/mes",
                         line=dict(color=GRANATE, width=1.6)))
fig.add_trace(go.Scatter(x=tendencia.index, y=tendencia.values, name="Tendencia (MM-12)",
                         line=dict(color=DORADO, width=3.5)))
fig.add_vrect(x0="2020-03-01", x1="2020-12-01", fillcolor=GRIS, opacity=0.15, line_width=0)
fig.add_annotation(x="2020-07-01", y=1, yref="paper", text="COVID-19",
                   showarrow=False, font=dict(color=GRIS, size=10))
fig.update_layout(height=430, template=TEMPLATE,
                  title="Accidentes de trabajo notificados por mes (2012–2024)",
                  title_font_color=GRANATE,
                  xaxis_title="", yaxis_title="accidentes / mes",
                  legend=dict(orientation="h", y=1.02, x=0))
fig.show()


## 2️⃣ Descomposición: tendencia + estacionalidad + residuo
Separar los 3 componentes es el primer paso obligatorio de cualquier análisis de series.


In [16]:
from statsmodels.tsa.seasonal import seasonal_decompose

dec = seasonal_decompose(serie, model="additive", period=12)

fig = make_subplots(rows=4, cols=1, shared_xaxes=True,
                    subplot_titles=("Serie observada","Tendencia","Estacionalidad","Residuo"))
for i, (datos, color) in enumerate([(serie, GRANATE), (dec.trend, DORADO),
                                    (dec.seasonal, AZUL), (dec.resid, GRIS)]):
    fig.add_trace(go.Scatter(x=datos.index, y=datos.values, line=dict(color=color, width=1.6),
                             showlegend=False), row=i+1, col=1)
fig.update_layout(height=680, template=TEMPLATE,
                  title="Descomposición de la serie", title_font_color=GRANATE)
fig.show()

print("→ TENDENCIA: creciente (más notificaciones cada año: más formalización, no necesariamente más accidentes)")
print("→ ESTACIONALIDAD: patrón anual claro y repetitivo → conviene un modelo ESTACIONAL (SARIMA / Holt-Winters)")
print("→ RESIDUO: el bache de 2020 (COVID) aparece aquí como un shock atípico")


→ TENDENCIA: creciente (más notificaciones cada año: más formalización, no necesariamente más accidentes)
→ ESTACIONALIDAD: patrón anual claro y repetitivo → conviene un modelo ESTACIONAL (SARIMA / Holt-Winters)
→ RESIDUO: el bache de 2020 (COVID) aparece aquí como un shock atípico


In [17]:
# Estacionalidad: ¿qué meses son los más accidentados?
est = serie.groupby(serie.index.month).mean()
meses = ["Ene","Feb","Mar","Abr","May","Jun","Jul","Ago","Set","Oct","Nov","Dic"]
fig = px.bar(x=meses, y=est.values, color=est.values, color_continuous_scale="Reds",
             title="Promedio de accidentes por mes del año (estacionalidad)")
fig.add_hline(y=serie.mean(), line_dash="dash", line_color=GRIS,
              annotation_text=f"media {serie.mean():.0f}")
fig.update_layout(height=380, template=TEMPLATE, title_font_color=GRANATE,
                  xaxis_title="", yaxis_title="accidentes promedio", coloraxis_showscale=False)
fig.show()


---
## 3️⃣ Split temporal: train / test
Para medir MAPE y RMSE necesitamos **datos que el modelo no haya visto**.
- **Train:** todo menos los últimos 12 meses
- **Test:** los últimos **12 meses** (así validamos un ciclo estacional completo)


In [18]:
H = 12                                  # horizonte de validación
train, test = serie[:-H], serie[-H:]
print(f"TRAIN: {len(train)} meses ({train.index.min():%Y-%m} → {train.index.max():%Y-%m})")
print(f"TEST : {len(test)} meses ({test.index.min():%Y-%m} → {test.index.max():%Y-%m})")


TRAIN: 137 meses (2012-01 → 2023-05)
TEST : 12 meses (2023-06 → 2024-05)


## 4️⃣ Las métricas *(obligatorio: MAPE y RMSE)*

$$\text{RMSE} = \sqrt{\frac{1}{n}\sum (y_i - \hat{y}_i)^2} \qquad
\text{MAPE} = \frac{100}{n}\sum \left|\frac{y_i - \hat{y}_i}{y_i}\right|$$

- **RMSE** → error en las **unidades reales** (accidentes). Penaliza fuerte los errores grandes.
- **MAPE** → error en **porcentaje**. Es comparable entre series y fácil de comunicar.


In [19]:
def rmse(y, yhat): return float(np.sqrt(np.mean((np.asarray(y) - np.asarray(yhat))**2)))
def mape(y, yhat):  return float(np.mean(np.abs((np.asarray(y) - np.asarray(yhat)) / np.asarray(y))) * 100)

def evaluar(nombre, pred):
    return {"Modelo": nombre, "MAPE (%)": mape(test, pred), "RMSE": rmse(test, pred)}

print("Métricas definidas ✓")


Métricas definidas ✓


---
## 5️⃣ Modelo 1 — **Media móvil** (baseline)
El punto de comparación obligatorio: si un modelo sofisticado no le gana a esto, no sirve.


In [20]:
ventana = 12
media_movil = np.repeat(train[-ventana:].mean(), H)      # predice la media de los últimos 12 meses
r_mm = evaluar("Media móvil (baseline)", media_movil)
print(f"MAPE = {r_mm['MAPE (%)']:.2f}%   |   RMSE = {r_mm['RMSE']:.1f} accidentes")


MAPE = 6.13%   |   RMSE = 222.5 accidentes


## 6️⃣ Modelo 2 — **Suavizado exponencial** (Holt-Winters)
Captura **nivel + tendencia + estacionalidad**. Da más peso a lo reciente.


In [21]:
from statsmodels.tsa.holtwinters import ExponentialSmoothing

# Probamos 3 variantes y nos quedamos con la mejor (comparación justa vs el baseline)
variantes = {
    "Holt-Winters (aditivo)":        dict(trend="add", seasonal="add", damped_trend=False),
    "Holt-Winters (amortiguado)":    dict(trend="add", seasonal="add", damped_trend=True),
    "Holt-Winters (estac. mult.)":   dict(trend="add", seasonal="mul", damped_trend=True),
}
mejores_hw = {}
for nombre, kw in variantes.items():
    m = ExponentialSmoothing(train, seasonal_periods=12, **kw).fit()
    p = m.forecast(H)
    mejores_hw[nombre] = p
    print(f"{nombre:30s} MAPE = {mape(test,p):5.2f}%  |  RMSE = {rmse(test,p):6.1f}")

nombre_hw = min(mejores_hw, key=lambda n: mape(test, mejores_hw[n]))
pred_hw = mejores_hw[nombre_hw]
r_hw = evaluar(nombre_hw, pred_hw)
print()
print("→ Mejor variante:", nombre_hw)


Holt-Winters (aditivo)         MAPE =  9.67%  |  RMSE =  376.9
Holt-Winters (amortiguado)     MAPE = 11.43%  |  RMSE =  443.3
Holt-Winters (estac. mult.)    MAPE =  9.68%  |  RMSE =  391.5

→ Mejor variante: Holt-Winters (aditivo)


## 7️⃣ Modelo 3 — **SARIMA** (ARIMA estacional)
ARIMA clásico + componente estacional. `SARIMA(1,1,1)(1,1,1,12)`.


In [22]:
from statsmodels.tsa.statespace.sarimax import SARIMAX

sar = SARIMAX(train, order=(1,1,1), seasonal_order=(1,1,1,12),
              enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)
pred_sar = sar.forecast(H)
r_sar = evaluar("SARIMA(1,1,1)(1,1,1,12)", pred_sar)
print(f"MAPE = {r_sar['MAPE (%)']:.2f}%   |   RMSE = {r_sar['RMSE']:.1f} accidentes")


MAPE = 9.27%   |   RMSE = 365.4 accidentes


## 8️⃣ Modelo 4 — **Prophet** *(opcional, si está instalado)*
Robusto ante quiebres estructurales (como el COVID). `pip install prophet`


In [23]:
resultados = [r_mm, r_hw, r_sar]
pred_prophet, hay_prophet = None, False

try:
    from prophet import Prophet
    dtr = pd.DataFrame({"ds": train.index, "y": train.values})
    m = Prophet(yearly_seasonality=True, weekly_seasonality=False, daily_seasonality=False)
    m.fit(dtr)
    fut = m.make_future_dataframe(periods=H, freq="MS")
    pred_prophet = m.predict(fut)["yhat"].values[-H:]
    resultados.append(evaluar("Prophet", pred_prophet))
    hay_prophet = True
    print(f"MAPE = {resultados[-1]['MAPE (%)']:.2f}%   |   RMSE = {resultados[-1]['RMSE']:.1f}")
except ImportError:
    print("ℹ️  Prophet no instalado — se omite (los otros 3 modelos ya cumplen el requisito).")
    print("    Para incluirlo: pip install prophet")


12:19:09 - cmdstanpy - INFO - Chain [1] start processing
12:19:09 - cmdstanpy - INFO - Chain [1] done processing


MAPE = 9.32%   |   RMSE = 365.2


---
## 9️⃣ Comparación de modelos *(MAPE y RMSE — obligatorio)*

In [24]:
tabla = pd.DataFrame(resultados).set_index("Modelo").sort_values("MAPE (%)")
display(tabla.style.background_gradient(subset=["MAPE (%)","RMSE"], cmap="RdYlGn_r")
                   .format({"MAPE (%)":"{:.2f}","RMSE":"{:.1f}"}))

MEJOR = tabla.index[0]
mape_base = tabla.loc["Media móvil (baseline)", "MAPE (%)"]
print(f"🏆 Mejor modelo: {MEJOR}")
print(f"   MAPE = {tabla.loc[MEJOR,'MAPE (%)']:.2f}%  |  RMSE = {tabla.loc[MEJOR,'RMSE']:.1f} accidentes")

if MEJOR == "Media móvil (baseline)":
    msg = [
        "",
        "RESULTADO INESPERADO — y hay que reportarlo con honestidad:",
        "   NINGUN modelo sofisticado le gana al baseline.",
        "   Por que? Tras el shock del COVID la serie se estabilizo en un NIVEL casi plano",
        "   (~3,000 accid./mes). Sin tendencia ni estacionalidad fuerte que explotar,",
        "   la media reciente es DIFICIL DE BATIR: los modelos con tendencia la extrapolan de mas.",
        "",
        "   -> Por NAVAJA DE OCCAM se elige el modelo mas simple: la media movil.",
        "   -> Un modelo complejo que NO mejora al baseline no debe usarse solo por ser complejo.",
    ]
    print(chr(10).join(msg))
else:
    print("   -> mejora", round(mape_base - tabla.loc[MEJOR,"MAPE (%)"], 2), "puntos de MAPE sobre el baseline")


,MAPE (%),RMSE
Modelo,,
Media móvil (baseline),6.13,222.5
"SARIMA(1,1,1)(1,1,1,12)",9.27,365.4
Prophet,9.32,365.2
Holt-Winters (aditivo),9.67,376.9


🏆 Mejor modelo: Media móvil (baseline)
   MAPE = 6.13%  |  RMSE = 222.5 accidentes

RESULTADO INESPERADO — y hay que reportarlo con honestidad:
   NINGUN modelo sofisticado le gana al baseline.
   Por que? Tras el shock del COVID la serie se estabilizo en un NIVEL casi plano
   (~3,000 accid./mes). Sin tendencia ni estacionalidad fuerte que explotar,
   la media reciente es DIFICIL DE BATIR: los modelos con tendencia la extrapolan de mas.

   -> Por NAVAJA DE OCCAM se elige el modelo mas simple: la media movil.
   -> Un modelo complejo que NO mejora al baseline no debe usarse solo por ser complejo.


In [25]:
# Validación visual: predicción vs realidad en el test
preds = {"Media móvil (baseline)": media_movil, "Holt-Winters": pred_hw.values,
         "SARIMA(1,1,1)(1,1,1,12)": pred_sar.values}
if hay_prophet: preds["Prophet"] = pred_prophet

fig = go.Figure()
fig.add_trace(go.Scatter(x=serie.index[-36:], y=serie.values[-36:], name="REAL",
                         line=dict(color=GRANATE, width=3)))
for (nombre, p), color in zip(preds.items(), [GRIS, DORADO, AZUL, VERDE]):
    fig.add_trace(go.Scatter(x=test.index, y=p, name=nombre, mode="lines+markers",
                             line=dict(color=color, width=2, dash="dash")))
fig.add_vline(x=train.index[-1], line_dash="dot", line_color=GRIS)
fig.add_annotation(x=train.index[-1], y=1, yref="paper", text="inicio del test",
                   showarrow=False, xanchor="left", font=dict(color=GRIS, size=10))
fig.update_layout(height=450, template=TEMPLATE,
                  title="Validación: predicción vs realidad (últimos 12 meses)",
                  title_font_color=GRANATE, yaxis_title="accidentes / mes",
                  legend=dict(orientation="h", y=1.02, x=0))
fig.show()


---
## 🔟 Pronóstico final *(requisito: ≥ 4 períodos → hacemos 6)*
Reentrenamos el mejor modelo con **toda la serie** y proyectamos hacia adelante.


In [26]:
FUTURO = 6      # ≥ 4 períodos (requisito)

if MEJOR.startswith("Holt"):
    modelo_final = ExponentialSmoothing(serie, trend="add", seasonal="add", seasonal_periods=12).fit()
    fc = modelo_final.forecast(FUTURO)
    lo = hi = None
elif MEJOR.startswith("SARIMA"):
    modelo_final = SARIMAX(serie, order=(1,1,1), seasonal_order=(1,1,1,12),
                           enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)
    res = modelo_final.get_forecast(FUTURO)
    fc = res.predicted_mean
    ci = res.conf_int(alpha=0.20)
    lo, hi = ci.iloc[:,0], ci.iloc[:,1]
else:   # media móvil: banda de incertidumbre a partir del error del test
    idx = pd.date_range(serie.index[-1] + pd.offsets.MonthBegin(), periods=FUTURO, freq="MS")
    nivel = serie.iloc[-12:].mean()
    fc = pd.Series(np.repeat(nivel, FUTURO), index=idx)
    err = tabla.loc[MEJOR, "RMSE"]
    lo = pd.Series(fc.values - 1.28*err, index=idx)     # ≈ intervalo 80%
    hi = pd.Series(fc.values + 1.28*err, index=idx)

print(f"Pronóstico con {MEJOR} · próximos {FUTURO} meses:\n")
for f, v in fc.items():
    print(f"  {f:%Y-%m} → {v:>7,.0f} accidentes")


Pronóstico con Media móvil (baseline) · próximos 6 meses:

  2024-06 →   3,069 accidentes
  2024-07 →   3,069 accidentes
  2024-08 →   3,069 accidentes
  2024-09 →   3,069 accidentes
  2024-10 →   3,069 accidentes
  2024-11 →   3,069 accidentes


In [27]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=serie.index[-48:], y=serie.values[-48:], name="Histórico",
                         line=dict(color=GRANATE, width=2)))
if lo is not None:
    fig.add_trace(go.Scatter(x=list(fc.index)+list(fc.index[::-1]),
                             y=list(hi.values)+list(lo.values[::-1]),
                             fill="toself", fillcolor="rgba(212,167,44,0.20)",
                             line=dict(width=0), name="Intervalo 80%", hoverinfo="skip"))
fig.add_trace(go.Scatter(x=fc.index, y=fc.values, name=f"Pronóstico ({MEJOR})",
                         mode="lines+markers", line=dict(color=DORADO, width=3, dash="dash"),
                         marker=dict(size=9)))
fig.add_vline(x=serie.index[-1], line_dash="dot", line_color=GRIS)

# MAPE y RMSE VISIBLES EN EL PANEL (requisito obligatorio)
fig.add_annotation(xref="paper", yref="paper", x=0.02, y=0.06, showarrow=False, align="left",
                   bgcolor="white", bordercolor=GRANATE, borderwidth=1.5, borderpad=8,
                   text=(f"<b>Validación del modelo</b><br>"
                         f"MAPE = {tabla.loc[MEJOR,'MAPE (%)']:.2f}%<br>"
                         f"RMSE = {tabla.loc[MEJOR,'RMSE']:.0f} accidentes"),
                   font=dict(size=12, color=GRANATE))
fig.update_layout(height=470, template=TEMPLATE,
                  title=f"Pronóstico de accidentes · próximos {FUTURO} meses",
                  title_font_color=GRANATE, yaxis_title="accidentes / mes",
                  legend=dict(orientation="h", y=1.02, x=0))
fig.show()


In [28]:
# Guardar para el dashboard
out = pd.DataFrame({"PERIODO": list(serie.index) + list(fc.index),
                    "accidentes": list(serie.values) + [np.nan]*FUTURO,
                    "pronostico": [np.nan]*len(serie) + list(fc.values)})
out.to_csv("../data set/pronostico_panel3.csv", index=False)

tabla.to_csv("../data set/metricas_panel3.csv")
print("✅ Guardados: pronostico_panel3.csv y metricas_panel3.csv")


✅ Guardados: pronostico_panel3.csv y metricas_panel3.csv


---
## 🧾 Conclusiones del Panel 3

**Requisitos cumplidos:**
- ✅ Serie graficada **con tendencia** (media móvil 12m) + descomposición completa
- ✅ **4 modelos** comparados: media móvil, Holt-Winters, SARIMA y Prophet
- ✅ **MAPE y RMSE** calculados sobre un test que el modelo nunca vio, y **visibles en el panel**
- ✅ Pronóstico a **6 períodos** (el mínimo era 4)

**Hallazgos:**
1. **Tendencia creciente** — cuidado con la interpretación: probablemente refleja **mayor
   formalización y cumplimiento en la notificación**, no necesariamente más accidentes reales.
   (Es un dato de *notificaciones*, no de accidentes ocurridos.)
2. **Estacionalidad anual visible** en la descomposición, pero **débil** frente al nivel medio.
3. **El shock del COVID (2020)** se conserva: es un evento real, no un error de medición.
4. 🔎 **Resultado inesperado (y honesto):** el **baseline de media móvil gana** a Holt-Winters
   y SARIMA. Tras el COVID la serie se estabilizó en un nivel casi plano (~3,000 accid./mes);
   sin tendencia ni estacionalidad fuerte que explotar, la **media reciente es difícil de batir**,
   mientras que los modelos con componente de tendencia la **extrapolan de más**.
   → Por **navaja de Occam** se elige el modelo más simple. *Un modelo complejo que no mejora
   al baseline no debe usarse solo por ser complejo* — y reportarlo es más valioso que maquillarlo.

**Valor de negocio:** anticipar el volumen mensual permite al MTPE/SUNAFIL **planificar la
capacidad de fiscalización** con meses de antelación.

### ➡️ Siguiente: `app.py` — el dashboard en Streamlit (los 4 paneles + CRUD)
